## Assignment: Data Splitting

1. Split your data into a training and test set
2. Use cross validation to fit a model on all numeric features. Report r2 values for each validation fold.
3. Fit your model on all of your training data and score on the test dataset. 

In [ ]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score as r2
from sklearn.metrics import mean_absolute_error as mae

computers = pd.read_csv(r"C:\BIT_Data_analyst\05_regressions\01_lesson\Computers.csv")

computers.tail()

,price,speed,hd,ram,screen,cd,multi,premium,ads,trend
6254,1690,100,528,8,15,no,no,yes,39,35
6255,2223,66,850,16,15,yes,yes,yes,39,35
6256,2654,100,1200,24,15,yes,no,yes,39,35
6257,2195,100,850,16,15,yes,no,yes,39,35
6258,2490,100,850,16,17,yes,no,yes,39,35


### Test Data Split

In [ ]:
from sklearn.model_selection import train_test_split

features = ["speed", "hd", "ram", "screen", "ads", "trend"]

X = sm.add_constant(computers[features])
y = computers["price"]

# Test Split Here
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Cross Validation Loop 

In [ ]:
# 1. Sukuriame KFold objektą
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 2. Sukuriame sąrašus rezultatams
cv_lm_r2s = []
cv_lm_mae = []

# 3. Cross-validation ciklas
for train_ind, val_ind in kf.split(X, y):
    # Train/validation padalinimas
    X_train, y_train = X.iloc[train_ind], y.iloc[train_ind]
    X_val, y_val = X.iloc[val_ind], y.iloc[val_ind]
    
    # Pridedame konstantą (statsmodels to reikia)
    X_train_const = sm.add_constant(X_train)
    X_val_const = sm.add_constant(X_val)
    
    # Modelio treniravimas
    model = sm.OLS(y_train, X_train_const).fit()
    
    # Prognozės
    preds = model.predict(X_val_const)
    
    # R2 ir MAE
    cv_lm_r2s.append(r2(y_val, preds))
    cv_lm_mae.append(mae(y_val, preds))

# 4. Rezultatai
print("All Validation R2s: ", [round(x, 3) for x in cv_lm_r2s])
print(f"Cross Val R2s: {round(np.mean(cv_lm_r2s), 3)} +- {round(np.std(cv_lm_r2s), 3)}")

print("All Validation MAEs: ", [round(x, 3) for x in cv_lm_mae])
print(f"Cross Val MAEs: {round(np.mean(cv_lm_mae), 3)} +- {round(np.std(cv_lm_mae), 3)}")


All Validation R2s:  [0.714, 0.707, 0.71, 0.715, 0.711]
Cross Val R2s: 0.711 +- 0.003
All Validation MAEs:  [223.835, 220.868, 233.451, 225.708, 224.973]
Cross Val MAEs: 225.767 +- 4.181


### Model Fit on All Training Data

In [4]:
# 1. Pridedame konstantą
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

# 2. Treniruojame modelį ant VISŲ train duomenų
final_model = sm.OLS(y_train, X_train_const).fit()

# 3. Prognozuojame ant test duomenų
test_preds = final_model.predict(X_test_const)

# 4. Skaičiuojame R² ir MAE
test_r2 = r2(y_test, test_preds)
test_mae = mae(y_test, test_preds)

print("Test R2:", round(test_r2, 3))
print("Test MAE:", round(test_mae, 3))


Test R2: 0.716
Test MAE: 223.692


### Score on Test Data

Test R2: 0.716
Test MAE: 223.692
Cross Val R2s: 0.711 +- 0.003
Cross Val MAEs: 225.767 +- 4.181

Modelio veikimas ant testiniu duomenu yra labai panasus i cross-validation, todel galime pasakyti jog modelis veikia stabiliai.